# 🐳 Day 01a: Docker for FastAPI — Production Containers

---

## 🎯 What You'll Master Today
- Dockerfile instructions and layer caching
- Multi-stage builds for smaller images
- .dockerignore and environment variables
- Production-ready FastAPI Dockerfile

---

## 🎭 The Shipping Container Analogy

Before shipping containers, cargo handling was chaos — different sizes, fragile items mixed with heavy ones. **Containers** standardized everything: same size, stackable, works on any ship.

Docker does the same for software: package your app with ALL dependencies into a **standardized container** that runs identically everywhere.

```
╔═══════════════════════════════════════════════════════════════════╗
║  DOCKER CORE CONCEPTS                                             ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Image       → Blueprint (read-only template)                   ║
║  Container   → Running instance of an image                     ║
║  Dockerfile  → Instructions to build an image                   ║
║  Layer       → Each instruction creates a cached layer          ║
║  Registry    → Where images are stored (Docker Hub, ECR)        ║
║                                                                   ║
║  Image : Container  =  Class : Instance  (OOP analogy)          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# Basic Dockerfile — Step by Step
# ============================================================

print("""
# Dockerfile

# 1. Base image (slim = smaller, no unnecessary tools)
FROM python:3.11-slim

# 2. Set working directory inside container
WORKDIR /app

# 3. Copy requirements first (layer caching!)
#    If requirements.txt didn't change → this layer is cached
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 4. Copy application code
COPY . .

# 5. Expose port (documentation, doesn't actually publish)
EXPOSE 8000

# 6. Run the app
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
""")

print("Layer caching strategy:")
print("  1. Things that change LEAST → top of Dockerfile")
print("  2. Things that change MOST → bottom of Dockerfile")
print("  3. requirements.txt changes rarely → copy before code")
print("  4. Source code changes often → copy last")

In [ ]:
# ============================================================
# Multi-Stage Build — Smaller Production Image
# ============================================================

print("""
# Dockerfile (multi-stage)

# ---- Stage 1: Builder ----
FROM python:3.11-slim AS builder

WORKDIR /app

# Install build dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    gcc libpq-dev && \\
    rm -rf /var/lib/apt/lists/*

# Create virtual environment
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt


# ---- Stage 2: Production ----
FROM python:3.11-slim

WORKDIR /app

# Install only runtime dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    libpq5 && \\
    rm -rf /var/lib/apt/lists/*

# Copy virtual env from builder (gcc and build tools NOT included)
COPY --from=builder /opt/venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Non-root user for security
RUN adduser --disabled-password --no-create-home appuser
USER appuser

COPY . .

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
""")

print("Multi-stage benefits:")
print("  Builder stage: has gcc, dev headers → needed for pip install")
print("  Production:    only runtime libs → much smaller image")
print("  Typical savings: 800MB → 200MB")

In [ ]:
# ============================================================
# .dockerignore — Exclude Unnecessary Files
# ============================================================

print("""
# .dockerignore
__pycache__
*.pyc
*.pyo
.git
.gitignore
.env
.venv
venv
*.md
docker-compose*.yml
Dockerfile*
.pytest_cache
.mypy_cache
htmlcov
tests/
docs/
*.log
""")

print("Without .dockerignore:")
print("  COPY . .  copies EVERYTHING — .git, venv, tests, docs")
print("  → Larger image, slower builds, potential secret leaks")
print("")
print("With .dockerignore:")
print("  Only application code gets copied")
print("  → Smaller image, faster builds, more secure")

In [ ]:
# ============================================================
# Environment Variables & Configuration
# ============================================================

print("""
# In Dockerfile — default values
ENV APP_ENV=production
ENV LOG_LEVEL=info

# Override at runtime:
docker run -e DATABASE_URL=postgres://... -e SECRET_KEY=abc123 myapp

# Or with env file:
docker run --env-file .env myapp

# In FastAPI — read from environment:
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    database_url: str
    secret_key: str
    debug: bool = False
    
    class Config:
        env_file = ".env"  # Fallback to .env file

settings = Settings()
""")

print("Docker commands cheat sheet:")
print("  docker build -t myapp:v1 .          → Build image")
print("  docker run -p 8000:8000 myapp:v1     → Run container")
print("  docker ps                            → List running")
print("  docker logs <id>                     → View logs")
print("  docker exec -it <id> /bin/sh         → Shell into container")
print("  docker images                        → List images")

In [ ]:
# ============================================================
# Production Dockerfile — Complete Template
# ============================================================

print("""
# Production Dockerfile for FastAPI

FROM python:3.11-slim AS builder
WORKDIR /app
RUN apt-get update && apt-get install -y --no-install-recommends \\
    gcc libpq-dev && rm -rf /var/lib/apt/lists/*
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

FROM python:3.11-slim
WORKDIR /app
RUN apt-get update && apt-get install -y --no-install-recommends \\
    libpq5 curl && rm -rf /var/lib/apt/lists/*
COPY --from=builder /opt/venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1
RUN adduser --disabled-password --no-create-home appuser
USER appuser
COPY --chown=appuser:appuser . .
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s \\
    CMD curl -f http://localhost:8000/health || exit 1
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
""")

print("Production checklist:")
print("  ✅ Multi-stage build")
print("  ✅ Non-root user")
print("  ✅ Health check")
print("  ✅ --no-cache-dir (smaller image)")
print("  ✅ PYTHONDONTWRITEBYTECODE + PYTHONUNBUFFERED")
print("  ✅ Multiple workers")
print("  ✅ .dockerignore")

---

## 🗺️ Docker Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  DOCKER FOR FastAPI                                               ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  BASE IMAGE:                                                     ║
║    python:3.11-slim   → Production (small)                      ║
║    python:3.11        → Dev (has build tools)                   ║
║    python:3.11-alpine → Smallest (but compatibility issues)     ║
║                                                                   ║
║  LAYER CACHE:                                                    ║
║    COPY requirements.txt → before COPY . .                      ║
║    Unchanged layers → instant rebuild                           ║
║                                                                   ║
║  SECURITY:                                                       ║
║    Non-root user (adduser)                                      ║
║    .dockerignore (no secrets in image)                          ║
║    No .env files baked into image                               ║
║                                                                   ║
║  CMD:                                                            ║
║    uvicorn app.main:app --host 0.0.0.0 --workers 4             ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Docker image vs container? | Image = blueprint (read-only). Container = running instance |
| 2 | What is layer caching? | Each instruction = layer. Unchanged layers reuse cache → fast rebuilds |
| 3 | Multi-stage builds? | Build deps in stage 1, copy only runtime to stage 2 → smaller image |
| 4 | Why non-root user? | If container compromised, attacker has limited permissions |
| 5 | COPY vs ADD? | COPY: simple file copy. ADD: also extracts archives, fetches URLs (prefer COPY) |
| 6 | CMD vs ENTRYPOINT? | CMD: default command (overridable). ENTRYPOINT: fixed command (args appended) |
| 7 | How to debug containers? | docker logs, docker exec -it <id> sh, docker inspect |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Dockerfile = recipe to build an image                │
## │  • COPY requirements.txt first → exploit layer cache    │
## │  • Multi-stage builds → smaller production images       │
## │  • Always use non-root user + .dockerignore             │
## │  • ENV for config, never bake secrets into images       │
## │  • HEALTHCHECK for container orchestration              │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Docker Compose** — Multi-container setups